In [ ]:
from bs4 import BeautifulSoup
from matplotlib.colors import ListedColormap

from geofeatureviz import helpers

There is an [official template provided by Wikipedia](https://upload.wikimedia.org/wikipedia/commons/b/b2/Maps_template-en.svg) for how maps should look. I downloaded this template, which is an SVG-file. Since SVG-files are in general XML-files, I could use BeautifulSoup to read it in. In Inkscape, I checked the group in which the rectangles are located, which has ID "g23682-6". Here, I take the fill colors of all rectangles in this group, sorted by its y-value. I printed the results in YAML-format and did some manual formatting to save it to map_style.yaml.

In [ ]:
wiki_template_path = (
    helpers.get_top_directory() / "data" / "Wikipedia_Maps_template-en.svg"
)

with wiki_template_path.open(encoding="utf-8") as f:
    soup = BeautifulSoup(f, "xml")

group = soup.find("g", {"id": "g23682-6"})
assert group is not None

fill_colors = []
for el in group.find_all():
    style = str(el.get("style"))
    style_dict = dict([s.split(":") for s in style.split(";")])
    fill = style_dict["fill"]
    y = el.get("y")

    fill_colors.append((y, fill))
fill_colors = [c[1] for c in sorted(fill_colors, key=lambda t: t[0])]

neutral_idx = fill_colors.index("#a7dfd2")  # neutral Wikipedia topo color from svg

topo_str = [f"topo:{i}" for i in range(neutral_idx, neutral_idx - len(fill_colors), -1)]
topo_to_color = dict(zip(topo_str, fill_colors))

# max len of current longest key in map_style.yaml for nice formatting
max_len = len("background")

for name, hexcode in topo_to_color.items():
    print(f'  {name}:{"":<{max_len - len(name)}} "{hexcode}"')

In [ ]:
cmap = ListedColormap(fill_colors)
cmap